In [6]:
%pip install pandas pyarrow
%pip install matplotlib


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from snowflake.snowpark import Session

session = Session.builder.configs({
    "account": "SFCOGSOPS-SNOWHOUSE_AWS_US_WEST_2",
    "user": "AALUSI",
    "authenticator": "externalbrowser",
    "warehouse": "AGAVIC_WH",
    "role": "PUBLIC",
    "database": "SNOWPUBLIC",
    "schema": "NOTEBOOKS",
}).create()

In [ ]:
df = session.sql("SELECT * FROM samples.tickit.sales LIMIT 1000").to_pandas()
df.head()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---- 0) Copy + basic dtype cleanup ----
df2 = df.copy()

# Ensure correct types (safe even if already correct)
for col in ["QTYSOLD", "PRICEPAID", "COMMISSION"]:
    df2[col] = pd.to_numeric(df2[col], errors="coerce")
df2["SALETIME"] = pd.to_datetime(df2["SALETIME"], errors="coerce")

# ---- 1) Derived fields ----
df2["GROSS_SALE"] = df2["PRICEPAID"]                    # total paid
df2["NET_SALE"]   = df2["PRICEPAID"] - df2["COMMISSION"]# after commission
df2["UNIT_PRICE"] = df2["PRICEPAID"] / df2["QTYSOLD"]
df2["SALE_DATE"]  = df2["SALETIME"].dt.date
df2["WEEK_START"] = df2["SALETIME"].dt.to_period("W").dt.start_time

# ---- 2) Core summaries (display these) ----
daily = (df2.groupby("SALE_DATE")
           .agg(orders=("SALESID","count"),
                qty=("QTYSOLD","sum"),
                gross=("GROSS_SALE","sum"),
                net=("NET_SALE","sum"))
           .reset_index())

top_sellers = (df2.groupby("SELLERID")
                 .agg(orders=("SALESID","count"),
                      qty=("QTYSOLD","sum"),
                      gross=("GROSS_SALE","sum"),
                      net=("NET_SALE","sum"),
                      avg_unit_price=("UNIT_PRICE","mean"))
                 .sort_values("gross", ascending=False)
                 .head(10)
                 .reset_index())

top_events = (df2.groupby("EVENTID")
                .agg(orders=("SALESID","count"),
                     qty=("QTYSOLD","sum"),
                     gross=("GROSS_SALE","sum"),
                     net=("NET_SALE","sum"))
                .sort_values("gross", ascending=False)
                .head(10)
                .reset_index())

price_qty_corr = df2[["QTYSOLD","UNIT_PRICE"]].corr().loc["QTYSOLD","UNIT_PRICE"]

print("📊 Daily summary:")
display(daily.head(10))

print("\n🏷️  Top sellers (by gross):")
display(top_sellers)

print("\n🎫  Top events (by gross):")
display(top_events)

print(f"\n🔗 Correlation (Qty vs Unit Price): {price_qty_corr:.3f}")

# ---- 3) Quick charts (matplotlib; no seaborn) ----

# 3a) Net revenue over time
plt.figure(figsize=(8,4))
plt.plot(daily["SALE_DATE"], daily["net"], marker="o")
plt.title("Net Revenue by Day")
plt.xlabel("Date")
plt.ylabel("Net Revenue")
plt.grid(True)
plt.tight_layout()
plt.show()

# 3b) Distribution of unit prices
plt.figure(figsize=(8,4))
df2["UNIT_PRICE"].dropna().plot(kind="hist", bins=20)
plt.title("Distribution of Unit Price")
plt.xlabel("Unit Price")
plt.ylabel("Count")
plt.grid(True)
plt.tight_layout()
plt.show()

# 3c) Quantity vs Unit Price scatter
plt.figure(figsize=(6,6))
plt.scatter(df2["QTYSOLD"], df2["UNIT_PRICE"])
plt.title("Qty vs Unit Price")
plt.xlabel("Quantity Sold")
plt.ylabel("Unit Price")
plt.grid(True)
plt.tight_layout()
plt.show()

# ---- 4) Optional: weekly view (nice for longer series) ----
weekly = (df2.groupby("WEEK_START")
            .agg(orders=("SALESID","count"),
                 qty=("QTYSOLD","sum"),
                 gross=("GROSS_SALE","sum"),
                 net=("NET_SALE","sum"))
            .reset_index())
print("\n📅 Weekly summary:")
display(weekly.head(10))

In [ ]:
!pip freeze > requirements.txt